In [1]:
from openai import OpenAI
from content_extracter import extract_text_and_links_from_page
import json

In [2]:
# Define Ollama as OpenAI object
OLLAMA_BASE_URL = "http://localhost:11434/v1"

ollama = OpenAI(base_url=OLLAMA_BASE_URL, api_key="ollama")

## Fetch important links from the home page of company

In [3]:
system_prompt_important_links = """
You are tasked to find out only the important links from a set of links mentioned in a company's website. The links importance are to be categorized from the fact
that they will be used to create a brochure for the company which will then be sent out as marketing strategies, hiring opportunities, investor readings, 
and so on.

The input given to you contains all links present in the home page of the company, each separated by a new line `\n`
The output you provide should follow the following JSON format - 
{
    links: [
        {"type": "about page", "url": "https://full.url/goes/here/about"},
        {"type": "careers page", "url": "https://another.full.url/careers"}
    ]
}
"""

In [4]:
def get_user_prompt_important_links(company_name, links):
    return f"""
    You are given the list of links from the home page of the company {company_name}. Filter out the list to give me only those links which are useful and valuable
    to be put up in the company's brochure. 
    The list of links are as follows:
    {links}
    """

In [14]:
async def fetch_content_and_important_links(company_name, page_url):
    text, links = await extract_text_and_links_from_page(page_url)
    print(f"Found a total of {len(links)} links from the home page of {company_name}")
    print("Extracting important links for brochure...")
    response = ollama.chat.completions.create(
        model="llama3.2:1b",
        messages=[
            {"role": "system", "content": system_prompt_important_links},
            {"role": "user", "content": get_user_prompt_important_links(company_name, links)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    try:
        important_links = json.loads(result) 
        print(f"Found {len(important_links['links'])} important links from homepage of {company_name}")
    except:
        print("Response from LLM not in json format")
        return

    return text, important_links

In [15]:
text, important_links = await fetch_content_and_important_links("Hugging Face", "https://huggingface.co")

Found a total of 77 links from the home page of Hugging Face
Extracting important links for brochure...
Found 3 important links from homepage of Hugging Face


In [16]:
important_links

{'links': [{'type': 'About page',
   'url': 'https://huggingface.co/hugging-face'},
  {'type': 'Careers page', 'url': 'https://huggingface.co/careers'},
  {'type': 'Enterprise Support', 'url': 'https://huggingface.co/support'}]}

## Create the brochure

In [ ]:
system_prompt_brochure_generator = """
You are tasked to create a detailed, useful, appealing and informative brochure from a company's home page and relevant links.
You'll be given, as an input, the entire content from the webpage as well as the important links related to the company. 
Do not hallucinate any piece of information and present only what is given to you in an appealing way.

The brochure will be sent out as marketing strategies, hiring opportunities, investor readings, and so on.
The format of your response (the brochure) should be completely outlined in the form of a markdown. 

Ensure you do not duplicate content generated. In other words, each section of the generated brochure should be unique. 
"""

In [28]:
def get_user_prompt_brochure_generator(company_name, home_page_contents, important_links):
    return f"""
        Take the following home page content and important links from the home page of {company_name}.
        Content: {home_page_contents}
        Important Links: {important_links}

        Create a brochure which outlines the company's important aspects that would make it appealing to the masses. Respond the brochure in the format of a markdown.
    """

In [29]:
prompt = get_user_prompt_brochure_generator("Hugging Face", text, important_links)
prompt

"\n        Take the following home page content and important links from the home page of Hugging Face.\n        Content: Hugging Face\nModels\nDatasets\nSpaces\nBuckets\nNEW\nDocs\nPricing\nWebsite\nTasks\nHuggingChat\nCollections\nLanguages\nOrganizations\nCommunity\nBlog\nPosts\nDaily Papers\nHardware\nLearn\nDiscord\nForum\nGitHub\nSolutions\nTeam & Enterprise\nHugging Face PRO\nEnterprise Support\nInference Providers\nInference Endpoints\nStorage Buckets\nLog In\nSign Up\nThe AI community building the future.\nThe platform where the machine learning community collaborates on models, datasets, and applications.\nExplore AI Apps\nor\nBrowse 2M+ models\nTrending on\nthis week\nModels\ndeepseek-ai/DeepSeek-V4.1-Flash-1017\nUpdated 5 days ago\n•\n51.9k\n•\n2.32k\nblack-forest-labs/flux-3-image-klein-comfy\nUpdated 5 days ago\n•\n513k\n•\n1.27k\nmeta-models/Muse-Spark-Open-816B-A42B\nUpdated 7 days ago\n•\n3.16k\n•\n653\nOpenMOSS-Team/MOSS-Image-SFT\nUpdated 1 day ago\n•\n6.89k\n•\n559\

In [30]:
async def create_brochure(company_name, page_url):
    content, important_links = await fetch_content_and_important_links(company_name, page_url)
    print("Creating brochure...")
    response = ollama.chat.completions.create(
        model="llama3.2:1b",
        messages=[
            {"role": "system", "content": system_prompt_brochure_generator},
            {"role": "user", "content": get_user_prompt_brochure_generator(company_name, content, important_links)}
        ]
    )
    print("Brochure created !")
    result = response.choices[0].message.content
    return result

In [31]:
brochure_content = await create_brochure("Hugging Face", "https://huggingface.co")

Found a total of 80 links from the home page of Hugging Face
Extracting important links for brochure...
Found 2 important links from homepage of Hugging Face
Creating brochure...
Brochure created !


In [33]:
from IPython.display import Markdown, display


display(Markdown(brochure_content))

# Hugging Face: The Future of AI
=====================================

## Introduction

Hugging Face is a leading platform for machine learning (ML) and artificial intelligence (AI) applications. With a vast collection of models, datasets, and tools, Hugging Face empowers developers to build, deploy, and collaborate on AI projects. In this brochure, we present the key aspects of Hugging Face that make it a go-to platform for the AI community.

## Models

Hugging Face offers a diverse selection of models for various applications, including:

* **Text and Image Models**: State-of-the-art models for tasks such as text-to-image, image-text, and image segmentation.
* **Language Models**: Large language models (LLMs) designed for language understanding and generation.
* **Graph Neural Networks (GNNs)**: Advanced GNN models for graph-structured data.
* **Reinforcement Learning (RL) Models**: Models for optimizing RL agents.

The Hugging Face Models library provides a wide range of models from various providers, including MoST, Hugging Face, and more. Models are updated regularly to reflect the latest advancements in ML.

## Datasets

Hugging Face's Datasets library offers a massive collection of labeled datasets for various AI applications, including:

* **Text Datasets**: Datasets for text classification, sentiment analysis, and question answering.
* **Image Datasets**: High-quality image datasets for image classification, object detection, and segmentation.
* **Audio Datasets**: datasets for speech and audio classification.

Datasets are curated and labeled to ensure high accuracy and relevance.

## Spaces

Hugging Face's Spaces is a decentralized storage system for storing models, datasets, and other assets. Spaces provides high-performance data storage, retrieval, and sharing capabilities.

## Pricing

Hugging Face offers various pricing plans for users, including:

* **Free**: Free for individuals and organizations with less than 10,000 models.
* **Pay-Per-Use**: Pay only for the usage time of your models.

## Inference Endpoints

Hugging Face provides access to a range of infrastructure components, including:

* **Inference Endpoints**: GPU-based models optimized for inference.
* **Model Servers**: In-memory servers for efficient data storage and retrieval.
* **Model Hubs**: Cloud-hosted model repositories for collaborative development.

Inference endpoints enable developers to deploy and query their models in various environments.

## Log In

For a seamless user experience, Hugging Face provides a single sign-on (SSO) system. SSO eliminates the need for multiple logins and improves productivity.

## Sign Up

Create an account on the Hugging Face platform to access our range of features, including:

* **Model Access**: Instant access to models, datasets, and APIs.
* **Collaboration**: Share knowledge and collaborate with others in real-time.
* **Community**: Join our growing community of ML enthusiasts and contributors.

## Team & Enterprise

For enterprises and organizations, Hugging Face offers a range of enterprise-grade services. These services include:

* **Model Hubs**: Custom-built model servers for enterprises with high scalability and performance requirements.
* **Pricing**: Customized pricing for enterprises requiring high volumes of data and usage.
* **Security**: Enterprise-grade security features, including access controls and audit logs.

## Careers

Hugging Face offers a range of career opportunities for enthusiasts and experts in the AI community. Explore our **Careers** section to learn more.

## Press
--------------------------------

Stay up-to-date with the latest news and research in AI by following our press releases on:

* **Twitter**: @HuggingFace
* **LinkedIn**: Hugging Face
* **GitHub**: HuggingFace

## Resources
----------------

For further learning and staying up-to-date with the latest ML trends, visit our **Resources** section, which includes:

* **Documentation**: Comprehensive documentation on our tools and platforms.
* **Blog**: Regular blog posts from industry experts and researchers.
* **Forum**: Live topics and discussions on our community forum.

## Learn
------------

Embark on your AI journey with Hugging Face!

### Learn Hugging Face

* **Hugging Face Hub**: Interact with our Python library and Hugging Face Hub.
* **Courses**: Official courses and tutorials on our platform.
* **Books**: Get hands-on experience with our tools and platforms.

### Build ML with Hugging Face

* **Modeling**: Create your own models using our PyTorch and TensorFlow APIs.
* **Deployment**: Deploy your models on our custom-built infrastructure.
* **Analysis**: Use our tools for data analysis and visualization.

## Documentation
-----------------

Hugging Face documentation is your ultimate Go-to-references manual for integrating with our platform.

* **Web API**: Learn how to work with our APIs and access data.
* **Python SDK**: Get started with our Python SDK and PyTorch Tensorflow APIs.
* **Community Notes**: Stay updated with our community notes, bug fixes, and releases.

## Social
------------

Join our community and share your knowledge and passion for AI!

* **Twitter**: Follow us on Twitter to stay up-to-date with the latest AI news.
* **LinkedIn**: Participate in our LinkedIn group and engage with our community.

## GitHub
------------

Contribute to our ongoing development of Hugging Face platforms and tools!

* **GitHub**: Explore our API documentation, code repositories, and contributed projects.
* **GitLab**: Explore our GitLab repository for ML and AI-related contributions.

## Press
------------

For media inquiries and expert voices, reach out to:

* **Twitter**: @HuggingFace
* **LinkedIn**: Hugging Face
* **GitHub**: HuggingFace

## Resources
----------------

Stay up-to-date with the latest news and research in AI!

* **Blog**: Our official blog, featuring news, research, and tutorials.
* **Forum**: Join our community discussions for AI enthusiasts and industry experts.
* **Tweet**: Stay updated on AI news and trends by following us.

## Learn
------------

Embark on your AI journey with Hugging Face!

### Learn Hugging Face

* **Hugging Face Hub**: Interact with our Python library and Hugging Face Hub.
* **Courses**: Official courses and tutorials on our platform.
* **Books**: Get hands-on experience with our tools and platforms.

### Build ML with Hugging Face

* **Modeling**: Create your own models using our PyTorch and TensorFlow APIs.
* **Deployment**: Deploy your models on our custom-built infrastructure.
* **Analysis**: Use our tools for data analysis and visualization.